<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/09_lightgbm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09: LightGBM per-crop models

**Purpose:** Replace the joint random forest by one LightGBM model per crop, trained on environment-only features, and compare with the baseline on SPAM agreement (this notebook) and on field observations (08b).

**Inputs:** `training_table.parquet`, `training_config.json` (07); `oof_rf_env.parquet` (08) for comparison.

**Method:**
1. Same pixel-level data and 5 spatial folds as notebook 08; environment-only features (51: climate, soil, terrain, salinity).
2. One LightGBM model per crop, on the share of harvested area, with the cross-entropy objective (labels in 0 to 1) and fixed hyperparameters. Missing values are handled natively. No agronomic constraints yet.
3. Out-of-fold predictions are saved for validation. Final models fitted on all pixels are used to measure size and prediction time.

**Outputs** (Google Drive): `oof_lgb_env.parquet`, `results_lgb.csv`, `lgb_env_models/` (one model file per crop).

**Results (out-of-fold, agreement with SPAM):**

| Method | Mean rank correlation | Top-5 overlap | Top-1 match | Distinct top-5 sets | Crops in top 5 in at least 1% of pixels |
|---|---|---|---|---|---|
| Observed SPAM | 1.000 | 1.000 | 1.000 | 430 | 16 |
| Random forest (08) | 0.644 | 0.789 | 0.591 | 147 | 13 |
| LightGBM per crop | 0.675 | 0.794 | 0.590 | 203 | 12 |

- Rank correlation improves for 12 of 16 crops (largest gains: rice, yams, soybean, tomato, sesame, onion) and is marginally lower for sorghum, groundnut, maize and cotton.
- Predicted magnitudes for rare crops are no better (R2 slightly lower for soybean, tomato, onion, sugarcane). Soybean, tomato, sweet potato and sugarcane almost never reach the top five by raw predicted share, because their shares are tiny everywhere; a relative score is tested in notebook 09b.
- All 16 models: 7.8 MB compressed (random forest: 53.9 MB); fitting 58 s; predicting 100,000 cells with all models 44 s (random forest: 3 s).

In [1]:
# @title Cell 1: load and aggregate
import os, json, time
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
P = f'{BASE}/crop_project'
cfg = json.load(open(f'{P}/training_config.json'))
FEATURES, GROUPS, CROPS = cfg['features'], cfg['groups'], cfg['crops']
ENV = GROUPS['climate'] + GROUPS['soil'] + GROUPS['terrain'] + GROUPS['salinity']
SHARE = [f'share_{c}' for c in CROPS]

tab = pd.read_parquet(f'{P}/training_table.parquet',
                      columns=['px', 'py', 'fold', 'adm0', 'has_label'] + ENV + SHARE)
lab = tab[tab.has_label]
g = lab.groupby(['px', 'py'])
pix = (g[ENV].mean().join(g[SHARE].first()).join(g[['fold']].first())
       .join(g['adm0'].agg(lambda s: s.mode().iat[0])).reset_index())
Y = np.clip(pix[SHARE].to_numpy('float64'), 0, 1)
folds = pix.fold.to_numpy()
print('Pixels:', len(pix), '| environment features:', len(ENV), '| crops:', len(CROPS))

Mounted at /content/drive
Pixels: 21128 | environment features: 51 | crops: 16


In [2]:
# @title Cell 2: metrics, baselines and the saved forest predictions
from scipy.stats import spearmanr
from sklearn.metrics import r2_score

K = 5
def top_mask(M, k=K):
    idx = np.argsort(-M, axis=1, kind='stable')[:, :k]
    m = np.zeros(M.shape, dtype=bool)
    np.put_along_axis(m, idx, True, axis=1)
    return m, idx

def overlap(Pr):
    ob, _ = top_mask(Y); pb, _ = top_mask(Pr)
    return (ob & pb).sum(1) / K

def summarize(Pr, name):
    pb, pi = top_mask(Pr)
    sets = pd.Series([tuple(sorted(r)) for r in pi]).value_counts()
    rho = np.nanmean([spearmanr(Y[:, j], Pr[:, j]).correlation for j in range(Y.shape[1])])
    return {'method': name, 'mean_spearman': round(float(rho), 3),
            'top5_overlap': round(float(overlap(Pr).mean()), 3),
            'top1_match': round(float((np.argmax(Pr, 1) == np.argmax(Y, 1)).mean()), 3),
            'distinct_top5_sets': len(sets),
            'most_common_set_share': round(float(sets.iloc[0] / len(Pr)), 3),
            'crops_in_top5_over_1pct': int((pb.mean(0) >= 0.01).sum())}

o = pd.read_parquet(f'{P}/oof_rf_env.parquet')
P_rf = pix[['px', 'py']].merge(o, on=['px', 'py'], how='left')[CROPS].to_numpy('float64')
Pg = np.zeros_like(Y); Pc = np.zeros_like(Y)
for k in range(5):
    tr, te = folds != k, folds == k
    gm = Y[tr].mean(0); Pg[te] = gm
    means = pd.DataFrame(Y[tr]).groupby(pix.adm0.to_numpy()[tr]).mean()
    cm = means.reindex(pix.adm0.to_numpy()[te])
    Pc[te] = np.where(cm.isna().to_numpy(), gm, cm.to_numpy())
print('RF predictions loaded:', P_rf.shape, '| missing:', int(np.isnan(P_rf).sum()))

RF predictions loaded: (21128, 16) | missing: 0


In [3]:
# @title Cell 3: LightGBM with spatial cross-validation
import lightgbm as lgb

PARAMS = dict(objective='cross_entropy', learning_rate=0.05, n_estimators=300, num_leaves=31,
              min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
              reg_lambda=5.0, n_jobs=-1, random_state=42, verbose=-1)
X = pix[ENV]
P_lgb = np.zeros_like(Y)
for j, c in enumerate(CROPS):
    t0 = time.time()
    for k in range(5):
        tr, te = folds != k, folds == k
        m = lgb.LGBMRegressor(**PARAMS).fit(X[tr], Y[tr, j])
        P_lgb[te, j] = m.predict(X[te])
    print(f'{c}: {time.time() - t0:.0f} s')

out = pix[['px', 'py', 'fold']].copy(); out[CROPS] = P_lgb
out.to_parquet(f'{P}/oof_lgb_env.parquet', index=False)

sorg: 16 s
pmil: 16 s
cowp: 16 s
grou: 16 s
maiz: 19 s
rice: 16 s
cass: 16 s
cott: 16 s
yams: 16 s
sesa: 16 s
onio: 15 s
soyb: 15 s
toma: 15 s
bean: 17 s
swpo: 17 s
sugc: 10 s


In [4]:
# @title Cell 4: results
rows = [summarize(Y, 'observed SPAM (reference)'), summarize(Pg, 'baseline: mean share'),
        summarize(Pc, 'baseline: mean share per country'),
        summarize(P_rf, 'RF: environment only'), summarize(P_lgb, 'LightGBM per crop: environment only')]
res = pd.DataFrame(rows).set_index('method')
pd.set_option('display.width', 220)
print(res.to_string())

def rho(Pr, j): return spearmanr(Y[:, j], Pr[:, j]).correlation
pc = pd.DataFrame({'mean_share': Y.mean(0),
                   'rho_rf': [rho(P_rf, j) for j in range(len(CROPS))],
                   'rho_lgb': [rho(P_lgb, j) for j in range(len(CROPS))],
                   'r2_rf': [r2_score(Y[:, j], P_rf[:, j]) for j in range(len(CROPS))],
                   'r2_lgb': [r2_score(Y[:, j], P_lgb[:, j]) for j in range(len(CROPS))]}, index=CROPS).round(3)
print('\nPer crop:'); print(pc)

freq = pd.DataFrame({'observed': top_mask(Y)[0].mean(0), 'RF': top_mask(P_rf)[0].mean(0),
                     'LightGBM': top_mask(P_lgb)[0].mean(0)}, index=CROPS).round(3)
print('\nShare of pixels where each crop is in the top 5:'); print(freq)
res.to_csv(f'{P}/results_lgb.csv')

                                     mean_spearman  top5_overlap  top1_match  distinct_top5_sets  most_common_set_share  crops_in_top5_over_1pct
method                                                                                                                                          
observed SPAM (reference)                    1.000         1.000       1.000                 430                  0.156                       16
baseline: mean share                        -0.054         0.733       0.311                   1                  1.000                        5
baseline: mean share per country             0.454         0.718       0.364                  12                  0.184                       10
RF: environment only                         0.644         0.789       0.591                 147                  0.168                       13
LightGBM per crop: environment only          0.675         0.794       0.590                 203                  0.192           

In [5]:
# @title Cell 5: model size and speed
import joblib
os.makedirs(f'{P}/lgb_env_models', exist_ok=True)
t0 = time.time(); final = {}
for j, c in enumerate(CROPS):
    final[c] = lgb.LGBMRegressor(**PARAMS).fit(X, Y[:, j])
    final[c].booster_.save_model(f'{P}/lgb_env_models/{c}.txt')
print(f'Final fit of {len(CROPS)} models: {time.time() - t0:.0f} s')
joblib.dump(final, '/content/lgb_tmp.joblib', compress=3)
print('Compressed size of all models (MB):', round(os.path.getsize('/content/lgb_tmp.joblib') / 1e6, 1))
Xc = tab.loc[tab.has_label, ENV].sample(100000, random_state=0)
t0 = time.time(); [m.predict(Xc) for m in final.values()]
print(f'Predicting 100,000 cells with all {len(CROPS)} models: {time.time() - t0:.0f} s')
os.remove('/content/lgb_tmp.joblib')
drive.flush_and_unmount(); drive.mount('/content/drive')

Final fit of 16 models: 58 s
Compressed size of all models (MB): 7.8
Predicting 100,000 cells with all 16 models: 44 s
Mounted at /content/drive
